# Preparación de datos para evaluación

Este notebook reúne las transformaciones necesarias de `DataUnderstanding` y `DataPreparation`. Si los ocho artefactos intermedios ya existen en `Datos/Procesado`, los carga directamente; si falta cualquiera, reconstruye los artefactos desde los datos crudos. Al final guarda las dos bases finales para modelado y evaluación.

No genera gráficos ni mensajes de consola. Los errores de lectura, transformación o escritura se propagan para evitar resultados parciales presentados como exitosos.


In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

RUTA_DATOS = Path("/content/drive/My Drive/Datos")
RUTA_CRUDO = RUTA_DATOS / "Crudo"
RUTA_PROCESADO = RUTA_DATOS / "Procesado"
RUTA_MAPEO = RUTA_DATOS / "mapeo_id_vaca.csv"

ARCHIVOS_INTERMEDIOS = {
    "consolidado_marzo_junio_2025": "df_consolidado",
    "concentrado_consumido": "concentradoConsumido",
    "produccion_leche_sesion": "prodLecheSesion",
    "s0_base_limpia": "s0",
    "s1_ventanas_robot": "s1",
    "s2_sesiones_etapa": "s2",
    "s3_episodios_cola": "s3",
    "s4_vaca_dia": "s4",
}

RUTA_PROCESADO.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [2]:
def ruta_intermedio(nombre):
    for extension in ("parquet", "csv"):
        ruta = RUTA_PROCESADO / f"{nombre}.{extension}"
        if ruta.is_file():
            return ruta
    return None


def leer_intermedio(ruta):
    if ruta.suffix == ".parquet":
        return pd.read_parquet(ruta)
    return pd.read_csv(ruta, low_memory=False)


def restaurar_tipos(dataframe):
    dataframe = dataframe.copy()
    if "mes" in dataframe.columns:
        dataframe["mes"] = pd.to_datetime(
            dataframe["mes"], errors="coerce"
        ).dt.to_period("M")
    if "fecha" in dataframe.columns:
        dataframe["fecha"] = pd.to_datetime(
            dataframe["fecha"], errors="coerce"
        )
    for columna in ("hora_inicio", "fin", "inicio", "ventana"):
        if columna in dataframe.columns:
            dataframe[columna] = pd.to_datetime(
                dataframe[columna], errors="coerce"
            )
    return dataframe


def preparar_guardado(dataframe):
    dataframe = dataframe.copy()
    for columna in dataframe.columns:
        if isinstance(dataframe[columna].dtype, pd.PeriodDtype):
            dataframe[columna] = dataframe[columna].astype(str)
    for columna in dataframe.select_dtypes(
        include=["object", "string"]
    ).columns:
        tipos = dataframe[columna].dropna().map(type).unique()
        if len(tipos) > 1:
            dataframe[columna] = dataframe[columna].astype("string")
    return dataframe


def exportar_intermedio(dataframe, nombre):
    dataframe = preparar_guardado(dataframe)
    ruta = RUTA_PROCESADO / f"{nombre}.parquet"
    dataframe.to_parquet(ruta, index=False)


rutas_intermedias = {
    nombre: ruta_intermedio(nombre) for nombre in ARCHIVOS_INTERMEDIOS
}
usar_intermedios_existentes = all(
    ruta is not None for ruta in rutas_intermedias.values()
)

if usar_intermedios_existentes:
    datos = {
        ARCHIVOS_INTERMEDIOS[nombre]: restaurar_tipos(leer_intermedio(ruta))
        for nombre, ruta in rutas_intermedias.items()
    }
else:
    datos = None


## Entendimiento de datos, solo cuando faltan artefactos

La reconstrucción usa las mismas carpetas de entrada y transformaciones base del notebook de entendimiento: consolida CSV y XLSX, anonimiza identificadores, calcula variables temporales y crea S0-S4. Se omiten únicamente las celdas exploratorias y visualizaciones.


In [3]:
def buscar_ruta_carpeta(nombre, raiz):
    nombre_buscar = nombre.casefold()
    for ruta_raiz, directorios, _ in os.walk(raiz):
        for directorio in directorios:
            if directorio.casefold() == nombre_buscar:
                return Path(ruta_raiz) / directorio
    raise FileNotFoundError(
        f"No se encontró la carpeta {nombre!r} dentro de {raiz}."
    )


def normalizar_id(serie):
    return serie.astype(str).str.extract(r"(\d+)", expand=False)


def actualizar_mapeo(ids):
    if RUTA_MAPEO.is_file():
        mapeo = pd.read_csv(RUTA_MAPEO, dtype=str)
    else:
        mapeo = pd.DataFrame(columns=["id_original", "id_anonimo"])

    nuevos = sorted(
        set(normalizar_id(pd.Series(ids)).dropna())
        - set(mapeo["id_original"])
    )
    if nuevos:
        np.random.default_rng().shuffle(nuevos)
        inicio = len(mapeo) + 1
        codigos = [
            f"VACA_{numero:03d}"
            for numero in range(inicio, inicio + len(nuevos))
        ]
        mapeo = pd.concat(
            [
                mapeo,
                pd.DataFrame(
                    {"id_original": nuevos, "id_anonimo": codigos}
                ),
            ],
            ignore_index=True,
        )
        mapeo.to_csv(RUTA_MAPEO, index=False)
    return dict(zip(mapeo["id_original"], mapeo["id_anonimo"]))


def anonimizar(dataframe):
    dataframe = dataframe.copy()
    mapeo = actualizar_mapeo(dataframe["id_vaca"])
    dataframe["id_vaca"] = normalizar_id(dataframe["id_vaca"]).map(mapeo)
    return dataframe.drop(columns=["origen_archivo"], errors="ignore")


def a_min(serie):
    valores = serie.astype(str).str.strip()
    valores = valores.where(~valores.isin(["nan", "NaT", "None", ""]))
    valores = valores.where(valores.str.count(":") != 1, "00:" + valores)
    valores = valores.str.split(" ").str[-1]
    return (
        pd.to_timedelta(valores, errors="coerce").dt.total_seconds() / 60
    )


def min_desde_alimento(tiempos):
    hora_decimal = tiempos.dt.hour + tiempos.dt.minute / 60
    ultimo_alimento = np.select(
        [hora_decimal >= 14.5, hora_decimal >= 7.5],
        [14.5, 7.5],
        default=14.5 - 24,
    )
    return ((hora_decimal - ultimo_alimento) * 60).round(1)


In [4]:
def construir_entendimiento():
    carpeta_csv = buscar_ruta_carpeta(
        "datos vacas en establo marzo_junio_2025", RUTA_CRUDO
    )
    archivos_csv = sorted(
        archivo
        for archivo in carpeta_csv.iterdir()
        if archivo.is_file() and archivo.suffix.casefold() == ".csv"
    )
    if not archivos_csv:
        raise FileNotFoundError(f"No hay CSV para procesar en {carpeta_csv}.")

    columnas_csv = [
        "hora_inicio", "accion", "duracion", "produccion_kg",
        "numero_ordeños", "rcs_celulas_ml", "patada", "incompleto",
        "pezones_no_encontrados", "estado_amd_ubre", "estado_amd_pezon",
        "media_flujos_di", "media_flujos_dd", "media_flujos_ti",
        "media_flujos_td", "sangre_di", "sangre_dd", "sangre_ti",
        "sangre_td", "conductividad_di", "conductividad_dd",
        "conductividad_ti", "conductividad_td", "eo_po", "usuario",
        "destino_leche", "razon_de_desviacion", "flujo_maximo_di",
        "flujo_maximo_dd", "flujo_maximo_ti", "flujo_maximo_td",
        "produccion_di", "produccion_dd", "produccion_ti", "produccion_td",
    ]
    lista_df = []
    for archivo in archivos_csv:
        if not archivo.name[0].isdigit():
            continue
        dataframe = pd.read_csv(archivo, skiprows=2, names=columnas_csv)
        dataframe["origen_archivo"] = archivo.name
        dataframe["id_vaca"] = archivo.stem
        lista_df.append(dataframe)
    if not lista_df:
        raise ValueError(f"No se encontraron CSV de vacas en {carpeta_csv}.")

    df_consolidado = anonimizar(pd.concat(lista_df, ignore_index=True))
    df_consolidado["hora_inicio"] = pd.to_datetime(
        df_consolidado["hora_inicio"], errors="coerce"
    )
    df_consolidado = df_consolidado.sort_values(
        "hora_inicio"
    ).reset_index(drop=True)

    raiz_historico = RUTA_CRUDO / "Datos vacas historico hasta enero 2026"
    carpeta_historico = buscar_ruta_carpeta("TEC-UAQ-UNAM", raiz_historico)
    archivos_excel = sorted(
        archivo
        for archivo in carpeta_historico.iterdir()
        if archivo.is_file() and archivo.suffix.casefold() in {".xlsx", ".xls"}
    )
    if not archivos_excel:
        raise FileNotFoundError(
            f"No hay archivos Excel para procesar en {carpeta_historico}."
        )

    columnas_consumo = ["alimento", "hora_evento", "consumo", "consumido"]
    columnas_produccion = [
        "hora_inicio", "numero_ordenio", "duracion", "produccion_kg",
        "rcs_celulas_ml", "intervalo_ordenio", "produccion_di",
        "produccion_dd", "produccion_ti", "produccion_td", "estado_md_ubre",
        "estado_md_pezon", "media_flujo_di", "media_flujo_dd",
        "media_flujo_ti", "media_flujo_td", "flujo_maximo_di",
        "flujo_maximo_dd", "flujo_maximo_ti", "flujo_maximo_td", "sangre_di",
        "sangre_dd", "sangre_ti", "sangre_td", "conductividad_di",
        "conductividad_dd", "conductividad_ti", "conductividad_td",
        "contador_ubre", "contador_di", "contador_dd", "contador_td",
        "contador_ti", "destino_leche", "razon_desviacion", "programa_lavado",
        "ms", "usuario", "patada", "incompleto", "pezones_no_encotnrados",
        "mdi",
    ]
    lista_concentrado = []
    lista_produccion = []
    for archivo in archivos_excel:
        id_vaca = "".join(filter(str.isdigit, archivo.name)) or "Desconocido"
        if archivo.name.casefold().startswith("concentrado consumido"):
            dataframe = pd.read_excel(
                archivo, skiprows=1, names=columnas_consumo
            )
            destino = lista_concentrado
        else:
            dataframe = pd.read_excel(
                archivo, skiprows=2, names=columnas_produccion
            )
            destino = lista_produccion
        dataframe["origen_archivo"] = archivo.name
        dataframe["id_vaca"] = id_vaca
        destino.append(dataframe)
    if not lista_concentrado or not lista_produccion:
        raise ValueError(
            "El histórico debe incluir archivos de concentrado y de "
            "producción de leche por sesión."
        )

    concentrado_consumido = anonimizar(
        pd.concat(lista_concentrado, ignore_index=True)
    )
    prod_leche_sesion = anonimizar(
        pd.concat(lista_produccion, ignore_index=True)
    )

    df = prod_leche_sesion.copy()
    df["hora_inicio"] = pd.to_datetime(df["hora_inicio"], errors="coerce")
    df["duracion_min"] = a_min(df["duracion"])
    df["intervalo_min"] = a_min(df["intervalo_ordenio"])
    df["fecha"] = df["hora_inicio"].dt.date
    df["hora"] = df["hora_inicio"].dt.hour
    df["dia_semana"] = df["hora_inicio"].dt.day_of_week
    df["mes"] = df["hora_inicio"].dt.to_period("M")

    vida_vaca = df.groupby("id_vaca")["hora_inicio"].agg(
        primer_registro="min", ultimo_registro="max"
    )
    inicio_datos = df["hora_inicio"].min()
    if pd.isna(inicio_datos):
        raise ValueError("No hay fechas válidas en los datos históricos.")

    umbral_secado_dias = 40
    lac = df.sort_values(["id_vaca", "hora_inicio"]).copy()
    lac["dias_sin_ordeño"] = (
        lac.groupby("id_vaca")["hora_inicio"].diff().dt.total_seconds()
        / 86400
    )
    es_primero = lac["dias_sin_ordeño"].isna()
    llegada_conocida = (
        lac["id_vaca"].map(vida_vaca["primer_registro"])
        > inicio_datos + pd.Timedelta(days=30)
    )
    nuevo_segmento = es_primero | (
        lac["dias_sin_ordeño"] >= umbral_secado_dias
    )
    lac["segmento"] = nuevo_segmento.groupby(lac["id_vaca"]).cumsum()
    lac["inicio_conocido"] = (lac["segmento"] > 1) | llegada_conocida
    lac["inicio_lactancia"] = lac.groupby(
        ["id_vaca", "segmento"]
    )["hora_inicio"].transform("min")
    lac["dias_en_leche"] = np.where(
        lac["inicio_conocido"],
        (lac["hora_inicio"] - lac["inicio_lactancia"]).dt.days,
        np.nan,
    )
    lac["etapa_lactancia"] = pd.cut(
        lac["dias_en_leche"],
        bins=[-1, 100, 200, 305, np.inf],
        labels=[
            "fresca (0-100)", "alta prod. (100-200)",
            "baja prod. (200-305)", "extendida (> 305)",
        ],
    )
    lac["grupo_vaca"] = np.select(
        [
            (lac["segmento"] == 1) & llegada_conocida,
            lac["segmento"] > 1,
        ],
        ["primer parto (probable)", "multipara"],
        default="desconocido",
    )

    min_duracion, max_duracion = 1, 30
    umbral_cola_min = 2
    umbral_prolongada_min = 45
    ventana = "30min"
    s0 = df.dropna(
        subset=["hora_inicio", "ms", "duracion_min"]
    ).copy()
    filtros = [
        ~s0["duracion_min"].between(min_duracion, max_duracion),
        s0["produccion_kg"] <= 0,
        s0["intervalo_min"] == 0,
    ]
    s0 = s0[~np.logical_or.reduce(filtros)]
    s0 = s0.sort_values(["ms", "hora_inicio"])
    s0["fin"] = s0["hora_inicio"] + pd.to_timedelta(
        s0["duracion_min"] * 60, unit="s"
    ).dt.round("s")
    s0["libre_previo_min"] = (
        s0["hora_inicio"] - s0.groupby("ms")["fin"].shift(1)
    ).dt.total_seconds() / 60
    s0 = s0[~(s0["libre_previo_min"] < -1)].copy()
    s0["año"] = s0["hora_inicio"].dt.year
    s0["fin_de_semana"] = s0["dia_semana"] >= 5
    s0["min_desde_alimento"] = min_desde_alimento(s0["hora_inicio"])
    for columna in ("dias_en_leche", "etapa_lactancia", "grupo_vaca"):
        s0[columna] = lac[columna].reindex(s0.index)
    s0["es_primer_parto"] = (
        (s0["grupo_vaca"] == "primer parto (probable)").astype(float)
        .where(s0["grupo_vaca"].isin(["primer parto (probable)", "multipara"]))
    )
    etapas = {
        "fresca": "fresca (0-100)",
        "alta": "alta prod. (100-200)",
        "baja": "baja prod. (200-305)",
        "extendida": "extendida (> 305)",
    }
    for nombre, etapa in etapas.items():
        s0[f"es_{nombre}"] = (
            (s0["etapa_lactancia"] == etapa)
            .astype(float)
            .where(s0["etapa_lactancia"].notna())
        )
    s0["vacas_activas_dia"] = s0.groupby("fecha")[
        "id_vaca"
    ].transform("nunique")

    calendario_horas = pd.date_range(
        df["hora_inicio"].min().floor("h"),
        df["hora_inicio"].max().floor("h"),
        freq="h",
    )
    ordenios_hora = (
        df.dropna(subset=["ms"])
        .assign(h=lambda frame: frame["hora_inicio"].dt.floor("h"))
        .groupby(["h", "ms"])
        .size()
        .unstack("ms", fill_value=0)
        .reindex(calendario_horas, fill_value=0)
    )
    sin_actividad_total = ordenios_hora.index[
        (ordenios_hora == 0).all(axis=1)
    ]

    s0["continua"] = s0["libre_previo_min"].between(-1, umbral_cola_min)
    s0["id_episodio"] = (~s0["continua"]).cumsum()
    s0["posicion_episodio"] = s0.groupby("id_episodio").cumcount()
    s3 = s0.groupby("id_episodio").agg(
        ms=("ms", "first"),
        inicio=("hora_inicio", "min"),
        fin=("fin", "max"),
        ordeños=("id_vaca", "size"),
        pct_primer_parto=("es_primer_parto", "mean"),
        pct_fresca=("es_fresca", "mean"),
        pct_alta=("es_alta", "mean"),
        pct_baja=("es_baja", "mean"),
        pct_extendida=("es_extendida", "mean"),
        vacas_activas_dia=("vacas_activas_dia", "first"),
    )
    columnas_pct = [
        "pct_primer_parto", "pct_fresca", "pct_alta",
        "pct_baja", "pct_extendida",
    ]
    s3[columnas_pct] = (s3[columnas_pct] * 100).round(1)
    s3["vacas_en_cola"] = s3["ordeños"] - 1
    s3 = s3[s3["vacas_en_cola"] >= 1].copy()
    s3["duracion_episodio_min"] = (
        (s3["fin"] - s3["inicio"]).dt.total_seconds() / 60
    )
    s3["prolongada"] = (
        s3["duracion_episodio_min"] >= umbral_prolongada_min
    )
    s3["fecha"] = s3["inicio"].dt.normalize()
    s3["hora"] = s3["inicio"].dt.hour
    s3["dia_semana"] = s3["inicio"].dt.dayofweek
    s3["año"] = s3["inicio"].dt.year
    s3["min_desde_alimento"] = min_desde_alimento(s3["inicio"])
    s0["entro_en_cola"] = (
        (s0["posicion_episodio"] > 0) & s0["id_episodio"].isin(s3.index)
    )
    s0["entro_en_cola_prolongada"] = (
        s0["entro_en_cola"]
        & s0["id_episodio"].map(s3["prolongada"]).eq(True)
    )

    malla = pd.MultiIndex.from_product(
        [
            sorted(s0["ms"].unique()),
            pd.date_range(
                s0["hora_inicio"].min().floor(ventana),
                s0["fin"].max().floor(ventana),
                freq=ventana,
            ),
        ],
        names=["ms", "ventana"],
    )

    def ventanas_de(episodios):
        inicio = episodios["inicio"].dt.floor(ventana)
        cantidad = (
            (episodios["fin"].dt.floor(ventana) - inicio)
            / pd.Timedelta(ventana)
        ).astype(int) + 1
        repeticiones = np.repeat(np.arange(len(episodios)), cantidad.values)
        desplazamiento = (
            np.concatenate([np.arange(k) for k in cantidad.values])
            if len(cantidad)
            else np.array([], dtype=int)
        )
        return pd.MultiIndex.from_arrays(
            [
                episodios["ms"].values[repeticiones],
                inicio.values[repeticiones]
                + desplazamiento * pd.Timedelta(ventana),
            ],
            names=["ms", "ventana"],
        ).unique()

    s1 = pd.DataFrame(index=malla)
    s1["cola"] = s1.index.isin(ventanas_de(s3)).astype(int)
    s1["cola_prolongada"] = s1.index.isin(
        ventanas_de(s3[s3["prolongada"]])
    ).astype(int)
    s0["ventana"] = s0["hora_inicio"].dt.floor(ventana)
    composicion = s0.groupby(["ms", "ventana"]).agg(
        ordeños=("id_vaca", "size"),
        minutos_ordeño=("duracion_min", "sum"),
        pct_primer_parto=("es_primer_parto", "mean"),
        pct_fresca=("es_fresca", "mean"),
        pct_alta=("es_alta", "mean"),
        pct_baja=("es_baja", "mean"),
        pct_extendida=("es_extendida", "mean"),
    )
    composicion[columnas_pct] = (composicion[columnas_pct] * 100).round(1)
    s1 = s1.join(composicion)
    s1["ordeños"] = s1["ordeños"].fillna(0).astype(int)
    s1["minutos_ordeño"] = s1["minutos_ordeño"].fillna(0)
    s1 = s1.reset_index()
    s1["fecha"] = s1["ventana"].dt.normalize()
    s1["hora"] = s1["ventana"].dt.hour
    s1["dia_semana"] = s1["ventana"].dt.dayofweek
    s1["fin_de_semana"] = s1["dia_semana"] >= 5
    s1["mes"] = s1["ventana"].dt.to_period("M")
    s1["año"] = s1["ventana"].dt.year
    s1["min_desde_alimento"] = min_desde_alimento(s1["ventana"])
    s1["vacas_activas_dia"] = s1["fecha"].map(
        s0.groupby(s0["hora_inicio"].dt.normalize())["id_vaca"].nunique()
    )
    s1 = s1.sort_values(["ms", "ventana"])
    s1["ocupacion_ventana_anterior_pct"] = (
        s1.groupby("ms")["minutos_ordeño"].shift(1)
        / (pd.Timedelta(ventana).total_seconds() / 60)
        * 100
    ).clip(upper=100)
    horas_corte = s1["ventana"].dt.floor("h").isin(sin_actividad_total)
    s1 = s1[~horas_corte].drop(
        columns=["minutos_ordeño"]
    ).reset_index(drop=True)

    columnas_s2 = [
        "id_vaca", "ms", "hora_inicio", "fecha", "mes", "año", "hora",
        "dia_semana", "fin_de_semana", "min_desde_alimento",
        "dias_en_leche", "etapa_lactancia", "grupo_vaca", "duracion_min",
        "produccion_kg", "vacas_activas_dia", "id_episodio", "entro_en_cola",
        "entro_en_cola_prolongada",
    ]
    s2 = s0.dropna(subset=["dias_en_leche"])[columnas_s2].copy()
    s2["etapa_lactancia"] = s2["etapa_lactancia"].cat.remove_unused_categories()
    s4 = s0.groupby(["id_vaca", "fecha"]).agg(
        ordeños=("hora_inicio", "size"),
        produccion_kg=("produccion_kg", "sum"),
        duracion_media_min=("duracion_min", "mean"),
        veces_en_cola=("entro_en_cola", "sum"),
        veces_en_cola_prolongada=("entro_en_cola_prolongada", "sum"),
        dias_en_leche=("dias_en_leche", "min"),
        etapa_lactancia=("etapa_lactancia", "first"),
        grupo_vaca=("grupo_vaca", "first"),
        vacas_activas_dia=("vacas_activas_dia", "first"),
    ).reset_index()
    s4["pct_en_cola"] = (
        s4["veces_en_cola"] / s4["ordeños"] * 100
    ).round(1)
    s4["pct_en_cola_prolongada"] = (
        s4["veces_en_cola_prolongada"] / s4["ordeños"] * 100
    ).round(1)
    s4["mes"] = pd.to_datetime(s4["fecha"]).dt.to_period("M")

    artefactos = {
        "consolidado_marzo_junio_2025": df_consolidado,
        "concentrado_consumido": concentrado_consumido,
        "produccion_leche_sesion": df,
        "s0_base_limpia": s0,
        "s1_ventanas_robot": s1,
        "s2_sesiones_etapa": s2,
        "s3_episodios_cola": s3.reset_index(),
        "s4_vaca_dia": s4,
    }
    return {
        ARCHIVOS_INTERMEDIOS[nombre]: restaurar_tipos(dataframe)
        for nombre, dataframe in artefactos.items()
    }, artefactos


if not usar_intermedios_existentes:
    datos, artefactos_intermedios = construir_entendimiento()
    for nombre, dataframe in artefactos_intermedios.items():
        exportar_intermedio(dataframe, nombre)

df_consolidado = datos["df_consolidado"]
concentradoConsumido = datos["concentradoConsumido"]
prodLecheSesion = datos["prodLecheSesion"]
s0, s1, s2, s3, s4 = (
    datos[nombre] for nombre in ("s0", "s1", "s2", "s3", "s4")
)


## Preparación de datos

Se reproducen las reglas de limpieza de sesiones y conflictos temporales, las etiquetas por intervalo, la agregación por robot y bloque de 30 minutos, el filtrado por cobertura y la construcción de las dos bases finales.


In [5]:
UMBRAL_COLA_MIN = 2
VENTANA = "30min"
MIN_VENTANA = pd.Timedelta(VENTANA).total_seconds() / 60
HORAS_ALIMENTO = [7.5, 14.5]
MESES_EXCLUIR = ["2022-07", "2026-01"]
UMBRAL_DIA = 0.5
VENTANA_DIAS = 30
MIN_ORDENIOS_BLOQUE = 1


def temporada_de_mes(mes):
    if mes in (12, 1, 2):
        return "invierno"
    if mes in (3, 4, 5):
        return "primavera"
    if mes in (6, 7, 8):
        return "verano"
    return "otoño"


pezones_clave = ["TD", "DI", "TI", "DD"]


def procesar_evento_pezones(dataframe, columna, prefijo):
    def analizar_registro(valor):
        if pd.isna(valor):
            return 0, {pezon: 0 for pezon in pezones_clave}
        valor = str(valor).upper().strip()
        if valor in {"NO", "FALSE", "0", "NONE", "N/A", ""}:
            return 0, {pezon: 0 for pezon in pezones_clave}
        if "TODOS" in valor:
            return 4, {pezon: 1 for pezon in pezones_clave}
        presencia = {
            pezon: int(pezon in valor) for pezon in pezones_clave
        }
        cantidad = sum(presencia.values())
        if cantidad == 0 and valor:
            cantidad = 1
        return cantidad, presencia

    resultado = dataframe[columna].apply(analizar_registro)
    dataframe[f"{prefijo}_cant"] = [fila[0] for fila in resultado]
    for pezon in pezones_clave:
        dataframe[f"{prefijo}_{pezon.lower()}"] = [
            fila[1][pezon] for fila in resultado
        ]
    return dataframe


consumo_clean = concentradoConsumido.copy()
produccion_clean = prodLecheSesion.copy()
consumo_clean.drop(columns="consumido", inplace=True)
produccion_clean.drop(columns=["rcs_celulas_ml", "usuario"], inplace=True)
produccion_clean["programa_lavado"] = (
    produccion_clean["programa_lavado"].eq("Local Aclarado").astype("boolean")
)
condicion = (
    produccion_clean["razon_desviacion"].isna()
    & (produccion_clean["destino_leche"] != "Tanque")
)
produccion_clean.loc[condicion, "razon_desviacion"] = "sin razón registrada"

produccion_clean = procesar_evento_pezones(
    produccion_clean, "patada", "patada"
)
produccion_clean = procesar_evento_pezones(
    produccion_clean, "incompleto", "incompleto"
)
produccion_clean = procesar_evento_pezones(
    produccion_clean, "pezones_no_encotnrados", "pezones_no_encontrados"
)

columnas_ceros = [
    columna for columna in produccion_clean.columns
    if columna.startswith(("conductividad_", "media_flujo_"))
]
produccion_clean[columnas_ceros] = produccion_clean[columnas_ceros].replace(
    0, np.nan
)
condicion_duracion = (
    (produccion_clean["duracion_min"] < 1)
    | (produccion_clean["duracion_min"] > 30)
)
produccion_clean["ordenio_anomalo"] = condicion_duracion.astype(int)
cuartos = ["produccion_di", "produccion_dd", "produccion_ti", "produccion_td"]
suma_cuartos = produccion_clean[cuartos].sum(axis=1, min_count=4)
produccion_clean["desbalance_produccion"] = (
    suma_cuartos.notna()
    & produccion_clean["produccion_kg"].notna()
    & ~np.isclose(
        produccion_clean["produccion_kg"], suma_cuartos, atol=0.1, rtol=0
    )
)
produccion_clean.dropna(subset=["ms"], inplace=True)

produccion_clean = produccion_clean.sort_values(
    ["ms", "hora_inicio"], kind="mergesort"
).copy()
produccion_clean["hora_fin"] = (
    produccion_clean["hora_inicio"]
    + pd.to_timedelta(produccion_clean["duracion_min"], unit="m")
)
indices_excluidos = []
minutos_traslape = {}
for maquina, grupo in produccion_clean.groupby("ms", sort=False):
    fin_ultimo_conservado = None
    for indice, fila in grupo.iterrows():
        inicio, fin = fila["hora_inicio"], fila["hora_fin"]
        if (
            fin_ultimo_conservado is not None
            and inicio < fin_ultimo_conservado
        ):
            indices_excluidos.append(indice)
            minutos_traslape[indice] = (
                fin_ultimo_conservado - inicio
            ).total_seconds() / 60
        else:
            fin_ultimo_conservado = fin
traslapes = produccion_clean.loc[indices_excluidos].copy()
traslapes["traslape_min"] = traslapes.index.map(minutos_traslape)
produccion_clean = produccion_clean.drop(index=indices_excluidos).copy()
produccion_clean.drop(columns=["hora_fin"], inplace=True)

produccion_clean_strict = produccion_clean.copy()
produccion_clean["fecha"] = pd.to_datetime(produccion_clean["fecha"])
mask_jul_2022 = (
    (produccion_clean["fecha"].dt.year == 2022)
    & (produccion_clean["fecha"].dt.month == 7)
)
mask_ene_2026 = (
    (produccion_clean["fecha"].dt.year == 2026)
    & (produccion_clean["fecha"].dt.month == 1)
)
produccion_clean_strict = produccion_clean[
    ~(mask_jul_2022 | mask_ene_2026)
].copy()
rango_dias_vaca = produccion_clean_strict.groupby("id_vaca")["fecha"].agg(
    lambda serie: (serie.max() - serie.min()).days
)
vacas_validas = rango_dias_vaca[rango_dias_vaca >= 30].index
produccion_clean_strict = produccion_clean_strict[
    produccion_clean_strict["id_vaca"].isin(vacas_validas)
].copy()

revision = produccion_clean.sort_values(["ms", "hora_inicio"]).copy()
revision["hora_fin_revision"] = (
    revision["hora_inicio"]
    + pd.to_timedelta(revision["duracion_min"], unit="m")
)
fin_previo = revision.groupby("ms")["hora_fin_revision"].transform(
    lambda serie: serie.cummax().shift()
)
solapamientos_restantes = int(
    (revision["hora_inicio"] < fin_previo).sum()
)
assert produccion_clean["ms"].notna().all(), "Persisten robots faltantes"
assert produccion_clean["hora_inicio"].notna().all(), "Persisten fechas faltantes"
assert produccion_clean["duracion_min"].gt(0).all(), "Persisten duraciones inválidas"
assert not produccion_clean.duplicated().any(), "Persisten duplicados exactos"
assert solapamientos_restantes == 0, "Persisten solapamientos temporales"

diario = (
    produccion_clean.assign(
        dia=produccion_clean["hora_inicio"].dt.normalize()
    )
    .groupby(["ms", "dia"])
    .size()
    .rename("ordenios_dia")
    .reset_index()
    .sort_values(["ms", "dia"])
)
mediana_robot = diario.groupby("ms")["ordenios_dia"].transform("median")
diario["mediana_movil"] = (
    diario.groupby("ms")["ordenios_dia"]
    .transform(
        lambda serie: serie.rolling(
            VENTANA_DIAS, min_periods=7, center=True
        ).median()
    )
    .fillna(mediana_robot)
)
diario["dia_completo"] = (
    diario["ordenios_dia"] >= UMBRAL_DIA * diario["mediana_movil"]
)

s = produccion_clean.sort_values(["ms", "hora_inicio"]).copy()
s["fin"] = s["hora_inicio"] + pd.to_timedelta(
    s["duracion_min"], unit="m"
)
s["siguiente_inicio"] = s.groupby("ms")["hora_inicio"].shift(-1)
s["tiempo_libre_min"] = (
    s["siguiente_inicio"] - s["fin"]
).dt.total_seconds() / 60
s = s[s["tiempo_libre_min"].between(0, 24 * 60)].copy()
s["posible_cola"] = s["tiempo_libre_min"] <= UMBRAL_COLA_MIN
s["bloque_tiempo"] = s["hora_inicio"].dt.floor(VENTANA)
s["pct_cola_bloque"] = (
    s.groupby(["ms", "bloque_tiempo"])["posible_cola"].transform("mean")
    * 100
)
produccion_clean = s
produccion_clean["min_desde_alimento"] = min_desde_alimento(
    produccion_clean["hora_inicio"]
)
destino = produccion_clean["destino_leche"]
produccion_clean["desviacion_leche"] = np.where(
    destino.isna(),
    np.nan,
    (destino.astype(str).str.strip() != "Tanque").astype(float),
)
llaves = ["id_vaca", "ms", "hora_inicio"]
columnas_lactancia = [
    "dias_en_leche", "etapa_lactancia", "grupo_vaca", "es_primer_parto",
    "es_fresca", "es_alta", "es_baja", "es_extendida",
]
lac_s0 = s0[llaves + columnas_lactancia].copy()
lac_s0["hora_inicio"] = lac_s0["hora_inicio"].astype("datetime64[ns]")
lac_s0 = lac_s0.drop_duplicates(subset=llaves).set_index(llaves)
produccion_clean["hora_inicio"] = produccion_clean["hora_inicio"].astype(
    "datetime64[ns]"
)
produccion_clean = (
    produccion_clean.drop(columns=columnas_lactancia, errors="ignore")
    .join(lac_s0, on=llaves)
)
produccion_clean_strict = produccion_clean[
    produccion_clean.index.isin(produccion_clean_strict.index)
].copy()

df_sistema = (
    produccion_clean.groupby(["ms", "bloque_tiempo"])
    .agg(
        pct_posible_cola=("posible_cola", "mean"),
        total_ordenios=("id_vaca", "count"),
        produccion_promedio=("produccion_kg", "mean"),
        duracion_promedio_min=("duracion_min", "mean"),
        duracion_total_min=("duracion_min", "sum"),
        patadas_totales=("patada_cant", "sum"),
        incompletos_totales=("incompleto_cant", "sum"),
        pezones_no_enc_totales=("pezones_no_encontrados_cant", "sum"),
        desviaciones_leche_totales=("desviacion_leche", "sum"),
        ordenios_con_lavado=("programa_lavado", "sum"),
        pct_primer_parto=("es_primer_parto", "mean"),
        pct_fresca=("es_fresca", "mean"),
        pct_alta=("es_alta", "mean"),
        pct_baja=("es_baja", "mean"),
        pct_extendida=("es_extendida", "mean"),
    )
    .reset_index()
)
df_sistema["pct_ocupacion_robot"] = (
    df_sistema["duracion_total_min"] / MIN_VENTANA
).clip(upper=1.0)
df_sistema["hora"] = df_sistema["bloque_tiempo"].dt.hour
df_sistema["dia_semana"] = df_sistema["bloque_tiempo"].dt.dayofweek
df_sistema["temporada"] = (
    df_sistema["bloque_tiempo"].dt.month.map(temporada_de_mes)
)
df_sistema["min_desde_alimento"] = min_desde_alimento(
    df_sistema["bloque_tiempo"]
)
df_sistema["ordenios_con_lavado"] = df_sistema[
    "ordenios_con_lavado"
].astype(int)
df_sistema["hubo_lavado"] = (
    df_sistema["ordenios_con_lavado"] > 0
).astype(int)
vacas_por_dia = produccion_clean.groupby(
    produccion_clean["hora_inicio"].dt.normalize()
)["id_vaca"].nunique()
df_sistema["vacas_activas_dia"] = (
    df_sistema["bloque_tiempo"].dt.normalize().map(vacas_por_dia)
)
df_sistema["ordenios_otros_robots"] = (
    df_sistema.groupby("bloque_tiempo")["total_ordenios"].transform("sum")
    - df_sistema["total_ordenios"]
)


In [6]:
malla = pd.MultiIndex.from_product(
    [
        sorted(df_sistema["ms"].unique()),
        pd.date_range(
            df_sistema["bloque_tiempo"].min(),
            df_sistema["bloque_tiempo"].max(),
            freq=VENTANA,
        ),
    ],
    names=["ms", "bloque_tiempo"],
)
completo = df_sistema.set_index(["ms", "bloque_tiempo"]).reindex(malla)
columnas_cero = [
    "total_ordenios", "duracion_total_min", "pct_ocupacion_robot"
]
completo[columnas_cero] = completo[columnas_cero].fillna(0)
grupos = completo.groupby(level="ms")
completo["ocupacion_bloque_anterior"] = grupos[
    "pct_ocupacion_robot"
].shift(1)
completo["ordenios_bloque_anterior"] = grupos["total_ordenios"].shift(1)
completo["pct_cola_bloque_anterior"] = (
    grupos["pct_posible_cola"]
    .shift(1)
    .where(completo["ordenios_bloque_anterior"] > 0, 0)
)
df_sistema = completo[
    completo["total_ordenios"] > 0
].reset_index()
columnas_enteras = [
    "total_ordenios", "patadas_totales", "incompletos_totales",
    "pezones_no_enc_totales", "ordenios_con_lavado", "hubo_lavado",
    "hora", "dia_semana", "vacas_activas_dia", "ordenios_otros_robots",
    "ordenios_bloque_anterior",
]
df_sistema[columnas_enteras] = df_sistema[columnas_enteras].astype("Int64")

df_sistema_sel = df_sistema.copy()
df_sistema_sel["dia"] = df_sistema_sel["bloque_tiempo"].dt.normalize()
df_sistema_sel = df_sistema_sel.merge(
    diario[["ms", "dia", "dia_completo"]],
    on=["ms", "dia"],
    how="left",
)
df_sistema_sel = df_sistema_sel[
    df_sistema_sel["dia_completo"].eq(True)
]
mes_bloque = df_sistema_sel["bloque_tiempo"].dt.to_period("M").astype(str)
df_sistema_sel = df_sistema_sel[
    ~mes_bloque.isin(MESES_EXCLUIR)
]
df_sistema_sel = df_sistema_sel[
    df_sistema_sel["total_ordenios"] >= MIN_ORDENIOS_BLOQUE
].drop(columns=["dia", "dia_completo"]).reset_index(drop=True)

columnas_etapa = [
    "pct_fresca", "pct_primer_parto", "pct_alta",
    "pct_baja", "pct_extendida",
]
df_sistema_sel["sin_etapa"] = (
    df_sistema_sel["pct_fresca"].isna().astype(int)
)
df_sistema_sel[columnas_etapa] = df_sistema_sel[
    columnas_etapa
].fillna(0)
df_sistema_sel = df_sistema_sel.sort_values(
    ["bloque_tiempo", "ms"]
).reset_index(drop=True)

IDENTIFICADORES = ["ms", "bloque_tiempo"]
OBJETIVO = "pct_posible_cola"
ENTRADAS = [
    columna
    for columna in df_sistema_sel.columns
    if columna not in IDENTIFICADORES + [OBJETIVO]
]
df_sistema_sel = df_sistema_sel[
    IDENTIFICADORES + ENTRADAS + [OBJETIVO]
]

EXCLUIDAS = [
    "incompletos_totales", "pezones_no_enc_totales",
    "duracion_promedio_min", "total_ordenios", "duracion_total_min",
    "pct_ocupacion_robot", "ordenios_otros_robots",
]
dataset_final = df_sistema_sel.drop(columns=EXCLUIDAS).copy()
dataset_final.to_parquet(
    RUTA_PROCESADO / "dataset_final_regresion.parquet",
    index=False,
)


In [7]:
llaves_ts = ["id_vaca", "ms", "hora_inicio"]
ordenios_ts = revision.copy()
ordenios_ts["hora_inicio"] = ordenios_ts["hora_inicio"].astype(
    "datetime64[ns]"
)
ordenios_ts = ordenios_ts.merge(
    produccion_clean[llaves_ts + ["posible_cola"]],
    on=llaves_ts,
    how="left",
    validate="one_to_one",
)
ordenios_ts["bloque_tiempo"] = ordenios_ts["hora_inicio"].dt.floor(VENTANA)
destino_ts = ordenios_ts["destino_leche"]
ordenios_ts["desviacion_leche"] = np.where(
    destino_ts.isna(),
    np.nan,
    (destino_ts.astype(str).str.strip() != "Tanque").astype(float),
)
mes_ts = ordenios_ts["bloque_tiempo"].dt.to_period("M").astype(str)
ordenios_ts = ordenios_ts.loc[
    ~mes_ts.isin(MESES_EXCLUIR)
].copy()

base_ts = ordenios_ts.groupby(IDENTIFICADORES).agg(
    pct_posible_cola=("posible_cola", "mean"),
    total_ordenios=("id_vaca", "size"),
    produccion_promedio=("produccion_kg", "mean"),
    duracion_total_min=("duracion_min", "sum"),
    patadas_totales=("patada_cant", "sum"),
    desviaciones_leche_totales=("desviacion_leche", "sum"),
    ordenios_con_lavado=("programa_lavado", "sum"),
)
columnas_etapa_ts = [
    "pct_primer_parto", "pct_fresca", "pct_alta",
    "pct_baja", "pct_extendida",
]
etapas_por_bloque_ts = df_sistema.set_index(IDENTIFICADORES)[
    columnas_etapa_ts
].copy()
base_ts = base_ts.join(
    etapas_por_bloque_ts, how="left", validate="one_to_one"
)

rangos_ts = ordenios_ts.groupby("ms")["bloque_tiempo"].agg(["min", "max"])
malla_ts = pd.MultiIndex.from_tuples(
    [
        (robot, bloque)
        for robot, rango in rangos_ts.iterrows()
        for bloque in pd.date_range(
            rango["min"], rango["max"], freq=VENTANA
        )
        if str(bloque.to_period("M")) not in MESES_EXCLUIR
    ],
    names=IDENTIFICADORES,
)
dataset_serie_tiempo = base_ts.reindex(malla_ts).reset_index()
sin_ordenios_ts = dataset_serie_tiempo["total_ordenios"].isna()
dataset_serie_tiempo["hubo_ordenios"] = (
    ~sin_ordenios_ts
).astype("int64")
columnas_cero_ts = [
    "total_ordenios", "duracion_total_min", "patadas_totales",
    "desviaciones_leche_totales", "ordenios_con_lavado",
    "pct_posible_cola",
]
dataset_serie_tiempo.loc[
    sin_ordenios_ts, columnas_cero_ts
] = 0
dataset_serie_tiempo["pct_posible_cola"] = dataset_serie_tiempo[
    "pct_posible_cola"
].astype(float)
dataset_serie_tiempo["pct_ocupacion_robot"] = (
    dataset_serie_tiempo["duracion_total_min"] / MIN_VENTANA
).clip(upper=1.0)
dataset_serie_tiempo["hubo_lavado"] = (
    dataset_serie_tiempo["ordenios_con_lavado"] > 0
).astype("int64")
dataset_serie_tiempo["ordenios_otros_robots"] = (
    dataset_serie_tiempo.groupby("bloque_tiempo")["total_ordenios"].transform(
        "sum"
    )
    - dataset_serie_tiempo["total_ordenios"]
)
tiempo_ts = dataset_serie_tiempo["bloque_tiempo"]
dataset_serie_tiempo["hora"] = tiempo_ts.dt.hour
dataset_serie_tiempo["dia_semana"] = tiempo_ts.dt.dayofweek
dataset_serie_tiempo["temporada"] = tiempo_ts.dt.month.map(
    temporada_de_mes
)
dataset_serie_tiempo["min_desde_alimento"] = min_desde_alimento(tiempo_ts)
vacas_dia_ts = ordenios_ts.groupby(
    ordenios_ts["hora_inicio"].dt.normalize()
)["id_vaca"].nunique()
dataset_serie_tiempo["vacas_activas_dia"] = (
    tiempo_ts.dt.normalize().map(vacas_dia_ts).fillna(0)
)
dataset_serie_tiempo["sin_etapa"] = (
    dataset_serie_tiempo["pct_fresca"].isna().astype("int64")
)
dataset_serie_tiempo[columnas_etapa_ts] = dataset_serie_tiempo[
    columnas_etapa_ts
].fillna(0)
columnas_enteras_ts = [
    "total_ordenios", "patadas_totales", "ordenios_con_lavado",
    "vacas_activas_dia", "ordenios_otros_robots",
]
dataset_serie_tiempo[columnas_enteras_ts] = dataset_serie_tiempo[
    columnas_enteras_ts
].astype("int64")
dataset_serie_tiempo = dataset_serie_tiempo.sort_values(
    ["bloque_tiempo", "ms"]
).reset_index(drop=True)
entradas_ts = [
    columna
    for columna in dataset_serie_tiempo.columns
    if columna not in IDENTIFICADORES + [OBJETIVO]
]
dataset_serie_tiempo = dataset_serie_tiempo[
    IDENTIFICADORES + entradas_ts + [OBJETIVO]
]
dataset_serie_tiempo.to_parquet(
    RUTA_PROCESADO / "dataset_final_serie_tiempo.parquet",
    index=False,
)
